In [3]:
import matplotlib
matplotlib.use('Agg')   # ✅ MUST come before pyplot

import matplotlib.pyplot as plt
import numpy as np

# ── All results in one place ──────────────────────────────────
cnn_results = {
    "BART-large-CNN":  {"r1":0.4475,"r2":0.2148,"rl":0.3079,"bleu":0.1102,"meteor":0.3444},
    "PEGASUS-large":   {"r1":0.4447,"r2":0.2138,"rl":0.3128,"bleu":0.1082,"meteor":0.3050},
    "T5-large":        {"r1":0.4176,"r2":0.1933,"rl":0.2939,"bleu":0.0756,"meteor":0.2776},
}

xsum_results = {
    "BART-large-CNN":  {"r1":0.2102,"r2":0.0356,"rl":0.1377,"bleu":0.0124,"meteor":0.1716},
    "PEGASUS-large":   {"r1":0.2200,"r2":0.0400,"rl":0.1475,"bleu":0.0150,"meteor":0.1645},
    "T5-large":        {"r1":0.2130,"r2":0.0345,"rl":0.1418,"bleu":0.0120,"meteor":0.1588},
}

models   = list(cnn_results.keys())
metrics  = ["r1","r2","rl","bleu"]
m_labels = ["ROUGE-1","ROUGE-2","ROUGE-L","BLEU"]

# ── FIGURE 1 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("CNN/DailyMail vs XSum Zero-Shot Transfer", fontsize=13, fontweight='bold')

x_pos = np.arange(len(models))
width = 0.35

for i, (metric, label) in enumerate(zip(metrics, m_labels)):
    ax = axes[i]

    cnn_vals  = [cnn_results[m][metric] for m in models]
    xsum_vals = [xsum_results[m][metric] for m in models]

    bars1 = ax.bar(x_pos - width/2, cnn_vals, width, label='CNN/DailyMail')
    bars2 = ax.bar(x_pos + width/2, xsum_vals, width, label='XSum')

    ax.set_title(label)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["BART","PEGASUS","T5-L"])
    ax.set_ylim(0, 0.55)

    for bar in bars1:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    if i == 0:
        ax.legend()

plt.tight_layout()
plt.savefig("figure1.png", dpi=300)
plt.close()
print("Figure 1 saved")

# ── FIGURE 2 ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

drop_keys = ["r1","r2","rl","bleu"]
drop_labels = ["ROUGE-1","ROUGE-2","ROUGE-L","BLEU"]

drops = {m: [] for m in models}

for model in models:
    for key in drop_keys:
        cnn_val  = cnn_results[model][key]
        xsum_val = xsum_results[model][key]
        drop = ((cnn_val - xsum_val) / cnn_val) * 100
        drops[model].append(drop)

x_pos = np.arange(len(drop_labels))
width = 0.25

for i, model in enumerate(models):
    bars = ax.bar(x_pos + (i-1)*width, drops[model], width, label=model)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

ax.axhline(y=50, linestyle='--', label='50% drop')

ax.set_title("Performance Drop (%)")
ax.set_ylabel("Drop %")
ax.set_xticks(x_pos)
ax.set_xticklabels(drop_labels)
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.savefig("figure2.png", dpi=300)
plt.close()
print("Figure 2 saved")

# ── FIGURE 3 ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

meteor_vals = [xsum_results[m]["meteor"] for m in models]

bars = ax.bar(models, meteor_vals)

for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{bar.get_height():.4f}', ha='center', va='bottom')

ax.set_title("METEOR Scores on XSum")
ax.set_ylabel("Score")
ax.set_ylim(0, 0.25)

plt.tight_layout()
plt.savefig("figure3.png", dpi=300)
plt.close()
print("Figure 3 saved")

# ── TABLE ───────────────────────────────────────────────────
print("\n" + "="*70)
print("RESULTS TABLE")
print("="*70)

for model in models:
    c = cnn_results[model]
    x = xsum_results[model]

    print(f"{model} (CNN): R1={c['r1']:.4f}, R2={c['r2']:.4f}, RL={c['rl']:.4f}, BLEU={c['bleu']:.4f}")
    print(f"{model} (XSum): R1={x['r1']:.4f}, R2={x['r2']:.4f}, RL={x['rl']:.4f}, BLEU={x['bleu']:.4f}, METEOR={x['meteor']:.4f}")

    drops = [( (c[k]-x[k])/c[k]*100 ) for k in drop_keys]
    print(f"Drop %: {[f'{d:.1f}%' for d in drops]}")
    print("-"*60)

Figure 1 saved
Figure 2 saved
Figure 3 saved

RESULTS TABLE
BART-large-CNN (CNN): R1=0.4475, R2=0.2148, RL=0.3079, BLEU=0.1102
BART-large-CNN (XSum): R1=0.2102, R2=0.0356, RL=0.1377, BLEU=0.0124, METEOR=0.1716
Drop %: ['53.0%', '83.4%', '55.3%', '88.7%']
------------------------------------------------------------
PEGASUS-large (CNN): R1=0.4447, R2=0.2138, RL=0.3128, BLEU=0.1082
PEGASUS-large (XSum): R1=0.2200, R2=0.0400, RL=0.1475, BLEU=0.0150, METEOR=0.1645
Drop %: ['50.5%', '81.3%', '52.8%', '86.1%']
------------------------------------------------------------
T5-large (CNN): R1=0.4176, R2=0.1933, RL=0.2939, BLEU=0.0756
T5-large (XSum): R1=0.2130, R2=0.0345, RL=0.1418, BLEU=0.0120, METEOR=0.1588
Drop %: ['49.0%', '82.2%', '51.8%', '84.1%']
------------------------------------------------------------
